# `libfec_parser` quickstart

`libfec_parser` is the Python binding for [libfec](https://github.com/asg017/libfec)'s Rust `.fec` parser. This notebook walks through both APIs it ships with:

1. **`libfec_parser.fecfile`** — a dict-based API modeled on the [`fecfile`](https://pypi.org/project/fecfile/) package. Rows come back as dicts keyed by column name. Start here.
2. **`libfec_parser.parser`** — a lower-level `Filing` class that gives you the raw positional fields of every row.

The package isn't on PyPI yet, so build it from source first. From `crates/fec-py/`, `make notebook` builds the wheel and opens this notebook with everything installed. See the [README](../README.md) for other options.

## Get a filing

Every electronic filing is a public `.fec` file at `https://docquery.fec.gov/dcdev/posted/<FILING_ID>.fec`. We'll use [FEC-1721696](https://docquery.fec.gov/cgi-bin/forms/C00016683/1721696/), Pfizer Inc. PAC's monthly report for July 2023: about 260 KB with ~1,400 itemized rows.

In [1]:
import urllib.request
from pathlib import Path

FILING_ID = 1721696
path = Path(f"{FILING_ID}.fec")

if not path.exists():
    url = f"https://docquery.fec.gov/dcdev/posted/{FILING_ID}.fec"
    with urllib.request.urlopen(url) as response:
        path.write_bytes(response.read())

print(f"{path}: {path.stat().st_size:,} bytes")

1721696.fec: 263,105 bytes


## The `fecfile` API

`from_file()` parses a filing into a plain dict with four keys:

- `header` — the `HDR` record: FEC format version and the software that produced the filing
- `filing` — the cover page (form type, committee, coverage dates, summary totals)
- `itemizations` — rows grouped by schedule: `"Schedule A"`, `"Schedule B"`, …
- `text` — free-form `TEXT` records

In [2]:
from libfec_parser import fecfile

parsed = fecfile.from_file(str(path))
parsed.keys()

dict_keys(['header', 'filing', 'itemizations', 'text'])

In [3]:
parsed["header"]

{'record_type': 'HDR',
 'ef_type': 'FEC',
 'fec_version': '8.4',
 'software_name': 'FECFile',
 'software_version': '8.4',
 'report_number': '0'}

The cover page has one key per column on the form. An F3X has more than a hundred, so here are just a few:

In [4]:
cover = parsed["filing"]
print(len(cover), "cover fields\n")

for key in [
    "form_type",
    "filer_committee_id_number",
    "committee_name",
    "report_code",
    "coverage_from_date",
    "coverage_through_date",
    "col_a_cash_on_hand_beginning_period",
    "col_a_total_receipts",
    "col_a_total_disbursements",
    "col_a_cash_on_hand_close_of_period",
]:
    print(f"{key:40} {cover[key]}")

123 cover fields

form_type                                F3XN
filer_committee_id_number                C00016683
committee_name                           PFIZER INC. PAC
report_code                              M8
coverage_from_date                       20230701
coverage_through_date                    20230731
col_a_cash_on_hand_beginning_period      394272.48
col_a_total_receipts                     83741.93
col_a_total_disbursements                57650.00
col_a_cash_on_hand_close_of_period       420364.41


> **Every value is a string.** Unlike the original `fecfile` package, amounts are not converted to floats and dates stay in the FEC's `YYYYMMDD` format. Convert them yourself, as shown in the pandas section below.

### Itemizations

Rows are grouped by schedule. The exact line number each row was reported on is in its `form_type` (`SA11AI`, `SB23`, …).

In [5]:
{schedule: len(rows) for schedule, rows in parsed["itemizations"].items()}

{'Schedule A': 1354, 'Schedule B': 33}

In [6]:
first = parsed["itemizations"]["Schedule A"][0]

# Only show the populated columns
{k: v for k, v in first.items() if v}

{'form_type': 'SA11AI',
 'filer_committee_id_number': 'C00016683',
 'transaction_id': '2023071716378-1066',
 'entity_type': 'IND',
 'contributor_last_name': 'Aaronson',
 'contributor_first_name': 'Eric',
 'contributor_street_1': '66 Hudson Blvd East',
 'contributor_city': 'New York',
 'contributor_state': 'NY',
 'contributor_zip_code': '10001',
 'contribution_date': '20230714',
 'contribution_amount': '104.17',
 'contribution_aggregate': '1458.38',
 'contributor_employer': 'Pfizer Inc',
 'contributor_occupation': 'SVP, Chief Counsel IP & IPE'}

### Into pandas

Each schedule is a list of dicts, which is exactly what `pd.DataFrame` wants.

In [7]:
import pandas as pd

receipts = pd.DataFrame(parsed["itemizations"]["Schedule A"])
receipts["contribution_amount"] = receipts["contribution_amount"].astype(float)
receipts["contribution_date"] = pd.to_datetime(receipts["contribution_date"], format="%Y%m%d")

receipts[
    [
        "contributor_last_name",
        "contributor_first_name",
        "contributor_state",
        "contributor_occupation",
        "contribution_date",
        "contribution_amount",
    ]
].head()

,contributor_last_name,contributor_first_name,contributor_state,contributor_occupation,contribution_date,contribution_amount
0,Aaronson,Eric,NY,"SVP, Chief Counsel IP & IPE",2023-07-14,104.17
1,Aaronson,Eric,NY,"SVP, Chief Counsel IP & IPE",2023-07-31,104.17
2,Aarts,Johanna,NY,"VP MTL, Rheumatology and New I&I Area",2023-07-14,20.84
3,Aarts,Johanna,NY,"VP MTL, Rheumatology and New I&I Area",2023-07-31,20.84
4,Adams,Dorinda,NY,National Pharmacy Business Manager,2023-07-14,20.00


In [8]:
total = receipts["contribution_amount"].sum()
print(f"{len(receipts):,} itemized receipts totaling ${total:,.2f}")
print(f"reported on the cover page:      ${float(cover['col_a_individuals_itemized']):,.2f}")

1,354 itemized receipts totaling $57,161.47
reported on the cover page:      $57,161.47


This PAC is funded by payroll deductions, so most people appear more than once in a month. Group by contributor to see who gave the most:

In [9]:
(
    receipts.groupby(["contributor_last_name", "contributor_first_name", "contributor_occupation"])["contribution_amount"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
    .head(10)
)

,,,count,sum
contributor_last_name,contributor_first_name,contributor_occupation,,
Susman,Sally,"Chief Corporate Affairs Officer, Execu",2,416.68
Dolsten,G.,"Chief Scientific Officer & President,",2,416.68
McDermott,Michael,"Chief Global Supply Officer, Executive",2,416.66
Power,Elizabeth,"Senior Director, Groton Site Affairs",2,416.66
Bishop-Murphy,Melissa,"Senior Director, State Government Rela",2,416.66
Mueller,Emily,Senior Director and Head of Congressio,2,416.66
Bourla,Albert,Chairman & CEO,2,416.66
Krebs,Matthew,"Senior Manager, Alliance Development",2,416.66
Johnson,Rady,"Chief Compliance,Quality & Risk Office",2,416.66


Schedule B is where the money goes. For a corporate PAC, that's mostly contributions to candidates and other committees.

In [10]:
disbursements = pd.DataFrame(parsed["itemizations"]["Schedule B"])
disbursements["expenditure_amount"] = disbursements["expenditure_amount"].astype(float)

(
    disbursements[["payee_organization_name", "payee_state", "expenditure_purpose_descrip", "expenditure_amount"]]
    .sort_values("expenditure_amount", ascending=False)
    .head(10)
)

,payee_organization_name,payee_state,expenditure_purpose_descrip,expenditure_amount
19,NRSC (Building Fund),DC,2023 Contribution,5000.0
7,DSCC (Building Fund),DC,2023 Contribution,5000.0
32,Republican Assembly Campaign Committee,WI,Nonfederal Contribution,3750.0
24,Committee to Elect a Republican Senate,WI,Nonfederal Contribution,3750.0
17,Mike Kelly For Congress,PA,2024 Primary,3000.0
11,Guy For Congress,PA,2024 Primary,2500.0
20,Oorah! Political Action Committee,IN,2023 Contribution,2500.0
1,Ann Wagner For Congress,MO,2024 Primary,2500.0
15,Lou Correa For Congress,CA,2024 Primary,2500.0
0,Alamo PAC,TX,2023 Contribution,2500.0


### Only parse what you need

`filter_itemizations` takes a list of row-type prefixes and drops everything else. On a large filing (ActBlue's reports run to several gigabytes) this saves most of the memory. An empty list skips itemizations entirely, leaving just the header and cover page.

In [11]:
only_sb = fecfile.from_file(str(path), options={"filter_itemizations": ["SB"]})
{schedule: len(rows) for schedule, rows in only_sb["itemizations"].items()}

{'Schedule B': 33}

In [12]:
cover_only = fecfile.loads(path.read_bytes(), options={"filter_itemizations": []})
cover_only["itemizations"]

{}

### Other entry points

- `loads(content)` parses `bytes`, a `str`, or a list of lines you already have in memory.
- `from_http(filing_id)` downloads from the FEC and parses in one step. It returns `None` if the filing doesn't exist.
- `parse_header(line)` and `parse_line(line, version)` parse a single record, if you're streaming a file yourself.

One gotcha when working with lines: `.fec` fields are separated by the ASCII 28 "file separator" character, which Python's `str.splitlines()` treats as a line break. Split on `"\n"` instead.

In [13]:
lines = path.read_text(encoding="latin-1").split("\n")

header, version, _ = fecfile.parse_header(lines[0])
print(header)

row = fecfile.parse_line(lines[2], version)
{k: v for k, v in row.items() if v}

{'record_type': 'HDR', 'ef_type': 'FEC', 'fec_version': '8.4', 'software_name': 'FECFile', 'software_version': '8.4', 'report_number': '0'}


{'form_type': 'SA11AI',
 'filer_committee_id_number': 'C00016683',
 'transaction_id': '2023071716378-1066',
 'entity_type': 'IND',
 'contributor_last_name': 'Aaronson',
 'contributor_first_name': 'Eric',
 'contributor_street_1': '66 Hudson Blvd East',
 'contributor_city': 'New York',
 'contributor_state': 'NY',
 'contributor_zip_code': '10001',
 'contribution_date': '20230714',
 'contribution_amount': '104.17',
 'contribution_aggregate': '1458.38',
 'contributor_employer': 'Pfizer Inc',
 'contributor_occupation': 'SVP, Chief Counsel IP & IPE'}

## The native `Filing` API

`libfec_parser.parser.Filing` is a thinner wrapper over the Rust parser. It accepts a path, `bytes`, or any file-like object with a `.read()` method (including an open `urlopen()` response).

In [14]:
from libfec_parser.parser import Filing

filing = Filing(str(path))
filing

Filing(form_type='F3XN', filer_id='C00016683', 1387 itemizations)

In [15]:
print(filing.header)
print(filing.header.fec_version, "|", filing.header.software_name, filing.header.software_version)
print()
print(filing.cover)
filing.cover.fields()

Header(fec_version='8.4', software_name='FECFile', software_version='8.4')
8.4 | FECFile 8.4

Cover(form_type='F3XN', filer_id='C00016683', filer_name='PFIZER INC. PAC')


{'form_type': 'F3XN',
 'filer_id': 'C00016683',
 'filer_name': 'PFIZER INC. PAC',
 'report_code': 'M8',
 'coverage_from_date': '2023-07-01',
 'coverage_through_date': '2023-07-31'}

Here the cover's coverage dates are ISO formatted (`2023-07-01`), and itemizations are positional: `row_type` plus a list of string fields, with no column names attached. It's the cheaper representation when you only need to count or filter rows.

In [16]:
from collections import Counter

Counter(item.row_type for item in filing.itemizations)

Counter({'SA11AI': 1354, 'SB23': 24, 'SB29': 9})

In [17]:
item = filing.itemizations[0]
print(item)
print(len(item), "fields")
print(item[0], "...", item[-1] or "(empty)")
item.fields()[:10]

Itemization(row_type='SA11AI', 45 fields)
45 fields
SA11AI ... (empty)


['SA11AI',
 'C00016683',
 '2023071716378-1066',
 '',
 '',
 'IND',
 '',
 'Aaronson',
 'Eric',
 '']

## Next steps

- The [libfec CLI](https://github.com/asg017/libfec) can export whole election cycles of filings to SQLite or CSV, which is usually the better tool when you need more than a handful of filings.
- Column names for every form and schedule come from libfec's mappings, across FEC format versions.